In [1]:
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: mps


In [4]:
X_train = np.load(
    "../data/processed/X_train_scaled.npy"
)

X_val = np.load(
    "../data/processed/X_val_scaled.npy"
)

X_test = np.load(
    "../data/processed/X_test_scaled.npy"
)

y_train = np.load(
    "../data/processed/y_train.npy"
)

y_val = np.load(
    "../data/processed/y_val.npy"
)

y_test = np.load(
    "../data/processed/y_test.npy"
)

dates_train = pd.read_csv(
    "../data/processed/dates_train.csv"
)

dates_val = pd.read_csv(
    "../data/processed/dates_val.csv"
)

dates_test = pd.read_csv(
    "../data/processed/dates_test.csv"
)

print("=" * 60)
print("LOADED STEP 3 DATA")
print("=" * 60)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

LOADED STEP 3 DATA
X_train: (124, 22, 9)
y_train: (124,)
X_val: (42, 22, 9)
y_val: (42,)
X_test: (42, 22, 9)
y_test: (42,)


In [5]:
assert X_train.shape == (124, 22, 9)
assert X_val.shape == (42, 22, 9)
assert X_test.shape == (42, 22, 9)

assert y_train.shape == (124,)
assert y_val.shape == (42,)
assert y_test.shape == (42,)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

print("Step 3 artifact validation: PASSED")

Step 3 artifact validation: PASSED


In [6]:
print("=" * 60)
print("LABEL DISTRIBUTIONS")
print("=" * 60)

print("\nTRAIN")
print(
    pd.Series(y_train)
    .map({0: "Fall", 1: "Rise"})
    .value_counts()
)

print("\nVALIDATION")
print(
    pd.Series(y_val)
    .map({0: "Fall", 1: "Rise"})
    .value_counts()
)

print("\nTEST")
print(
    pd.Series(y_test)
    .map({0: "Fall", 1: "Rise"})
    .value_counts()
)

LABEL DISTRIBUTIONS

TRAIN
Rise    63
Fall    61
Name: count, dtype: int64

VALIDATION
Rise    31
Fall    11
Name: count, dtype: int64

TEST
Rise    24
Fall    18
Name: count, dtype: int64


In [7]:
class TemporalBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        dilation=1,
        dropout=0.2
    ):
        super().__init__()

        padding = (
            kernel_size - 1
        ) * dilation

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(
            dropout
        )

        self.residual = (
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):

        residual = self.residual(x)

        out = self.conv1(x)

        out = out[:, :, :x.size(2)]

        out = self.relu(out)

        out = self.dropout(out)

        out = self.conv2(out)

        out = out[:, :, :x.size(2)]

        out = self.relu(out)

        out = self.dropout(out)

        return self.relu(
            out + residual
        )

In [8]:
class AAPLTCN(nn.Module):

    def __init__(
        self,
        num_features=9,
        num_classes=2
    ):
        super().__init__()

        self.tcn = nn.Sequential(

            TemporalBlock(
                in_channels=num_features,
                out_channels=32,
                kernel_size=3,
                dilation=1,
                dropout=0.2
            ),

            TemporalBlock(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                dilation=2,
                dropout=0.2
            )
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def forward(self, x):

        # Input:
        # (batch, timesteps, features)

        x = x.transpose(
            1,
            2
        )

        # (batch, features, timesteps)

        x = self.tcn(x)

        # Use final timestep representation

        x = x[:, :, -1]

        return self.classifier(x)

In [9]:
model = AAPLTCN(
    num_features=9,
    num_classes=2
).to(device)

print(model)

AAPLTCN(
  (tcn): Sequential(
    (0): TemporalBlock(
      (conv1): Conv1d(9, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (relu): ReLU()
      (dropout): Dropout(p=0.2, inplace=False)
      (residual): Conv1d(9, 32, kernel_size=(1,), stride=(1,))
    )
    (1): TemporalBlock(
      (conv1): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (relu): ReLU()
      (dropout): Dropout(p=0.2, inplace=False)
      (residual): Identity()
    )
  )
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)


In [10]:
total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("MODEL PARAMETERS")
print("=" * 60)

print(
    "Total parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)

MODEL PARAMETERS
Total parameters: 10594
Trainable parameters: 10594


In [11]:
dummy_input = torch.tensor(
    X_train[:4],
    dtype=torch.float32
).to(device)

with torch.no_grad():

    dummy_output = model(
        dummy_input
    )

print(
    "Input shape:",
    dummy_input.shape
)

print(
    "Output shape:",
    dummy_output.shape
)

assert dummy_output.shape == (
    4,
    2
)

print(
    "TCN forward-pass validation: PASSED"
)

Input shape: torch.Size([4, 22, 9])
Output shape: torch.Size([4, 2])
TCN forward-pass validation: PASSED


In [12]:
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 8
Validation batches: 3
Test batches: 3


In [13]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Loss: CrossEntropyLoss")
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Loss: CrossEntropyLoss
Optimizer: AdamW
Learning rate: 0.001
Weight decay: 0.0001


In [14]:
def run_epoch(model, loader, criterion, optimizer=None):

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    predictions = []
    targets = []

    for batch_X, batch_y in loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):

            logits = model(batch_X)

            loss = criterion(
                logits,
                batch_y
            )

            if is_training:
                loss.backward()

                optimizer.step()

        total_loss += (
            loss.item()
            * len(batch_y)
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        predictions.extend(
            preds.detach().cpu().numpy()
        )

        targets.extend(
            batch_y.detach().cpu().numpy()
        )

    average_loss = (
        total_loss
        / len(loader.dataset)
    )

    accuracy = accuracy_score(
        targets,
        predictions
    )

    balanced_accuracy = balanced_accuracy_score(
        targets,
        predictions
    )

    return (
        average_loss,
        accuracy,
        balanced_accuracy
    )

In [15]:
MAX_EPOCHS = 100
PATIENCE = 15

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

best_model_path = (
    "../models/aapl_tcn.pt"
)

history = []

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

print("Max epochs:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Best-model criterion: Validation Loss")
print("Model path:", best_model_path)

TRAINING CONFIGURATION
Max epochs: 100
Early stopping patience: 15
Best-model criterion: Validation Loss
Model path: ../models/aapl_tcn.pt


In [16]:
for epoch in range(1, MAX_EPOCHS + 1):

    train_loss, train_acc, train_balanced = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_acc, val_balanced = run_epoch(
        model,
        val_loader,
        criterion
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_balanced_accuracy": train_balanced,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_balanced_accuracy": val_balanced
    })

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            model.state_dict(),
            best_model_path
        )

    else:

        patience_counter += 1

    if patience_counter >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}"
        )

        break

Epoch 001 | Train Loss: 0.5786 | Train Acc: 0.694 | Val Loss: 1.2388 | Val Acc: 0.381
Epoch 002 | Train Loss: 0.5487 | Train Acc: 0.694 | Val Loss: 1.3238 | Val Acc: 0.381
Epoch 003 | Train Loss: 0.5221 | Train Acc: 0.718 | Val Loss: 1.4134 | Val Acc: 0.381
Epoch 004 | Train Loss: 0.4963 | Train Acc: 0.726 | Val Loss: 1.3258 | Val Acc: 0.357
Epoch 005 | Train Loss: 0.4859 | Train Acc: 0.766 | Val Loss: 1.4792 | Val Acc: 0.381
Epoch 006 | Train Loss: 0.4623 | Train Acc: 0.790 | Val Loss: 1.7021 | Val Acc: 0.381
Epoch 007 | Train Loss: 0.4268 | Train Acc: 0.831 | Val Loss: 1.6768 | Val Acc: 0.405
Epoch 008 | Train Loss: 0.4290 | Train Acc: 0.815 | Val Loss: 1.7376 | Val Acc: 0.429
Epoch 009 | Train Loss: 0.4004 | Train Acc: 0.815 | Val Loss: 1.7692 | Val Acc: 0.429
Epoch 010 | Train Loss: 0.3764 | Train Acc: 0.879 | Val Loss: 1.6747 | Val Acc: 0.500
Epoch 011 | Train Loss: 0.3753 | Train Acc: 0.879 | Val Loss: 1.9193 | Val Acc: 0.429
Epoch 012 | Train Loss: 0.3746 | Train Acc: 0.855 | Va

In [17]:
history_df = pd.DataFrame(history)

best_row = history_df.loc[
    history_df["val_loss"].idxmin()
]

print("=" * 60)
print("TRAINING HISTORY ANALYSIS")
print("=" * 60)

print(
    "Best epoch:",
    int(best_row["epoch"])
)

print(
    f"Best validation loss: "
    f"{best_row['val_loss']:.4f}"
)

print(
    f"Validation accuracy at best epoch: "
    f"{best_row['val_accuracy']:.4f}"
)

print(
    f"Validation balanced accuracy at best epoch: "
    f"{best_row['val_balanced_accuracy']:.4f}"
)

print(
    "\nFinal training accuracy:",
    f"{history_df.iloc[-1]['train_accuracy']:.4f}"
)

print(
    "Final validation accuracy:",
    f"{history_df.iloc[-1]['val_accuracy']:.4f}"
)

print(
    "\nSaved checkpoint corresponds to epoch:",
    int(best_row["epoch"])
)

TRAINING HISTORY ANALYSIS
Best epoch: 1
Best validation loss: 1.2388
Validation accuracy at best epoch: 0.3810
Validation balanced accuracy at best epoch: 0.5806

Final training accuracy: 0.8790
Final validation accuracy: 0.4524

Saved checkpoint corresponds to epoch: 1


In [18]:
print("\nFULL HISTORY")
print(
    history_df.to_string(index=False)
)


FULL HISTORY
 epoch  train_loss  train_accuracy  train_balanced_accuracy  val_loss  val_accuracy  val_balanced_accuracy
     1    0.578637        0.693548                 0.695290  1.238780      0.380952               0.580645
     2    0.548669        0.693548                 0.694770  1.323823      0.380952               0.551320
     3    0.522134        0.717742                 0.718579  1.413400      0.380952               0.551320
     4    0.496283        0.725806                 0.725995  1.325764      0.357143               0.505865
     5    0.485889        0.766129                 0.765678  1.479202      0.380952               0.551320
     6    0.462257        0.790323                 0.791049  1.702066      0.380952               0.551320
     7    0.426773        0.830645                 0.830991  1.676797      0.404762               0.567449
     8    0.428973        0.814516                 0.814858  1.737596      0.428571               0.583578
     9    0.400365     

In [19]:
baseline_history = history_df.copy()

baseline_best_epoch = int(
    baseline_history.loc[
        baseline_history["val_balanced_accuracy"].idxmax(),
        "epoch"
    ]
)

baseline_best_balanced_accuracy = (
    baseline_history["val_balanced_accuracy"].max()
)

print("=" * 60)
print("BASELINE EXPERIMENT")
print("=" * 60)

print(
    "Best validation balanced accuracy:",
    f"{baseline_best_balanced_accuracy:.4f}"
)

print(
    "Best epoch:",
    baseline_best_epoch
)

BASELINE EXPERIMENT
Best validation balanced accuracy: 0.6613
Best epoch: 10


In [20]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "../data/processed/tcn_training_history.csv",
    index=False
)

print("Training history saved.")

Training history saved.


In [21]:
assert os.path.exists(
    "../models/aapl_tcn.pt"
)

checkpoint = torch.load(
    "../models/aapl_tcn.pt",
    map_location=device,
    weights_only=True
)

print("=" * 60)
print("BEST MODEL CHECKPOINT")
print("=" * 60)

print("Model file exists: PASSED")
print("Best validation epoch:", best_epoch)
print(
    "Best validation loss:",
    f"{best_val_loss:.4f}"
)

BEST MODEL CHECKPOINT
Model file exists: PASSED
Best validation epoch: 1
Best validation loss: 1.2388


In [22]:
model.load_state_dict(
    torch.load(
        "../models/aapl_tcn.pt",
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("Best TCN checkpoint loaded successfully.")

Best TCN checkpoint loaded successfully.


In [23]:
test_predictions = []
test_probabilities = []
test_actual = []

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        batch_X = batch_X.to(device)

        logits = model(batch_X)

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

        test_actual.extend(
            batch_y.numpy()
        )

test_predictions = np.array(
    test_predictions
)

test_probabilities = np.array(
    test_probabilities
)

test_actual = np.array(
    test_actual
)

print(
    "Test predictions generated:",
    len(test_predictions)
)

assert len(test_predictions) == len(y_test)

print("Test inference validation: PASSED")

Test predictions generated: 42
Test inference validation: PASSED


In [24]:
test_accuracy = accuracy_score(
    test_actual,
    test_predictions
)

test_balanced_accuracy = balanced_accuracy_score(
    test_actual,
    test_predictions
)

test_precision = precision_score(
    test_actual,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_actual,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_actual,
    test_predictions,
    zero_division=0
)

print("=" * 60)
print("FINAL TCN TEST EVALUATION")
print("=" * 60)

print(f"Test samples:        {len(test_actual)}")
print(f"Accuracy:             {test_accuracy:.4f}")
print(f"Balanced Accuracy:    {test_balanced_accuracy:.4f}")
print(f"Precision:            {test_precision:.4f}")
print(f"Recall:               {test_recall:.4f}")
print(f"F1:                   {test_f1:.4f}")

FINAL TCN TEST EVALUATION
Test samples:        42
Accuracy:             0.4286
Balanced Accuracy:    0.5000
Precision:            0.0000
Recall:               0.0000
F1:                   0.0000


In [25]:
cm = confusion_matrix(
    test_actual,
    test_predictions,
    labels=[0, 1]
)

print("=" * 60)
print("TEST CONFUSION MATRIX")
print("=" * 60)

print("              Predicted")
print("              Fall  Rise")
print(
    f"Actual Fall   {cm[0,0]:4d}  {cm[0,1]:4d}"
)

print(
    f"Actual Rise   {cm[1,0]:4d}  {cm[1,1]:4d}"
)

TEST CONFUSION MATRIX
              Predicted
              Fall  Rise
Actual Fall     18     0
Actual Rise     24     0


In [26]:
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        test_actual,
        test_predictions,
        labels=[0, 1],
        target_names=["Fall", "Rise"],
        zero_division=0
    )
)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Fall       0.43      1.00      0.60        18
        Rise       0.00      0.00      0.00        24

    accuracy                           0.43        42
   macro avg       0.21      0.50      0.30        42
weighted avg       0.18      0.43      0.26        42



In [27]:
actual_distribution = pd.Series(
    test_actual
).map({
    0: "Fall",
    1: "Rise"
}).value_counts()

prediction_distribution = pd.Series(
    test_predictions
).map({
    0: "Fall",
    1: "Rise"
}).value_counts()

print("=" * 60)
print("TEST PREDICTION DISTRIBUTION")
print("=" * 60)

print("\nActual:")
print(actual_distribution)

print("\nPredicted:")
print(prediction_distribution)

TEST PREDICTION DISTRIBUTION

Actual:
Rise    24
Fall    18
Name: count, dtype: int64

Predicted:
Fall    42
Name: count, dtype: int64


In [28]:
train_majority_class = np.bincount(
    y_train
).argmax()

baseline_predictions = np.full(
    len(y_test),
    train_majority_class
)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

baseline_balanced_accuracy = balanced_accuracy_score(
    y_test,
    baseline_predictions
)

print("=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)

print(
    "Training majority class:",
    "Fall" if train_majority_class == 0 else "Rise"
)

print(
    f"Baseline Accuracy:          "
    f"{baseline_accuracy:.4f}"
)

print(
    f"Baseline Balanced Accuracy: "
    f"{baseline_balanced_accuracy:.4f}"
)

print(
    f"\nTCN Accuracy:                "
    f"{test_accuracy:.4f}"
)

print(
    f"TCN Balanced Accuracy:       "
    f"{test_balanced_accuracy:.4f}"
)

print(
    "\nTCN beats baseline:",
    test_accuracy > baseline_accuracy
)

BASELINE COMPARISON
Training majority class: Rise
Baseline Accuracy:          0.5714
Baseline Balanced Accuracy: 0.5000

TCN Accuracy:                0.4286
TCN Balanced Accuracy:       0.5000

TCN beats baseline: False


The TCN failed to outperform the naive majority-class baseline on the held-out test period. Although the model achieved substantially higher training performance, its test predictions collapsed to a single class, indicating poor generalization under the temporal distribution shift.

In [29]:
test_fall_probability = test_probabilities[:, 0]
test_rise_probability = test_probabilities[:, 1]

probability_df = pd.DataFrame({
    "Date": dates_test.iloc[:, 0],
    "Actual": test_actual,
    "Predicted": test_predictions,
    "Fall_Probability": test_fall_probability,
    "Rise_Probability": test_rise_probability
})

probability_df["Actual"] = probability_df["Actual"].map({
    0: "Fall",
    1: "Rise"
})

probability_df["Predicted"] = probability_df["Predicted"].map({
    0: "Fall",
    1: "Rise"
})

print("=" * 60)
print("TEST PREDICTION PROBABILITIES")
print("=" * 60)

print(
    probability_df.to_string(index=False)
)

TEST PREDICTION PROBABILITIES
      Date Actual Predicted  Fall_Probability  Rise_Probability
2026-06-12   Rise      Fall          0.891841          0.108159
2026-06-15   Rise      Fall          0.880521          0.119479
2026-06-16   Rise      Fall          0.876851          0.123150
2026-06-17   Fall      Fall          0.900127          0.099873
2026-06-18   Fall      Fall          0.892975          0.107025
2026-06-22   Fall      Fall          0.901109          0.098891
2026-06-23   Fall      Fall          0.914718          0.085282
2026-06-24   Fall      Fall          0.911132          0.088868
2026-06-25   Fall      Fall          0.914349          0.085651
2026-06-26   Rise      Fall          0.857566          0.142434
2026-06-29   Rise      Fall          0.838513          0.161487
2026-06-30   Rise      Fall          0.802048          0.197952
2026-07-01   Rise      Fall          0.820033          0.179968
2026-07-02   Rise      Fall          0.829686          0.170314
2026-07-06

In [30]:
print("=" * 60)
print("PROBABILITY SUMMARY")
print("=" * 60)

print(
    "Mean Fall probability:",
    f"{test_fall_probability.mean():.4f}"
)

print(
    "Mean Rise probability:",
    f"{test_rise_probability.mean():.4f}"
)

print(
    "Minimum Fall probability:",
    f"{test_fall_probability.min():.4f}"
)

print(
    "Maximum Fall probability:",
    f"{test_fall_probability.max():.4f}"
)

print(
    "Minimum Rise probability:",
    f"{test_rise_probability.min():.4f}"
)

print(
    "Maximum Rise probability:",
    f"{test_rise_probability.max():.4f}"
)

PROBABILITY SUMMARY
Mean Fall probability: 0.9322
Mean Rise probability: 0.0678
Minimum Fall probability: 0.8020
Maximum Fall probability: 0.9895
Minimum Rise probability: 0.0105
Maximum Rise probability: 0.1980


In [31]:
prediction_confidence = np.max(
    test_probabilities,
    axis=1
)

print("=" * 60)
print("PREDICTION CONFIDENCE")
print("=" * 60)

print(
    "Mean confidence:",
    f"{prediction_confidence.mean():.4f}"
)

print(
    "Minimum confidence:",
    f"{prediction_confidence.min():.4f}"
)

print(
    "Maximum confidence:",
    f"{prediction_confidence.max():.4f}"
)

PREDICTION CONFIDENCE
Mean confidence: 0.9322
Minimum confidence: 0.8020
Maximum confidence: 0.9895
